# Student Academic Performance Prediction

**Machine Learning project for Google Colab**

This notebook predicts a student's final Mathematics grade (`G3`, from 0 to 20) using study habits, school, family, and other available characteristics. To simulate prediction before the exam periods, `G1` and `G2` are excluded from the features.

**Dataset:** [UCI Student Performance Dataset](https://archive.ics.uci.edu/dataset/320/student%2Bperformance). The data contains student grades and demographic, social, and school-related attributes. Cite the dataset as: Cortez, P. (2008), UCI Machine Learning Repository, https://doi.org/10.24432/C5TG7T.


## 1. Import libraries


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

## 2. Load the dataset
The Mathematics dataset is loaded directly from a public dataset file. It uses semicolons as separators.


In [ ]:
DATA_URL = 'https://huggingface.co/datasets/DElmazi/Student_Performance/resolve/main/student-mat.csv'
df = pd.read_csv(DATA_URL, sep=';')

print('Dataset shape:', df.shape)
display(df.head())

## 3. Understand and clean the data


In [ ]:
print('Column names:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
df.info()
print('\nMissing values per column:')
display(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nDuplicate rows:', df.duplicated().sum())

# Remove duplicate rows if any are present
df = df.drop_duplicates().copy()
print('Shape after removing duplicates:', df.shape)

## 4. Exploratory Data Analysis (EDA)


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['G3'], bins=21, kde=True)
plt.title('Distribution of Final Mathematics Grades (G3)')
plt.xlabel('Final grade (0–20)')
plt.ylabel('Number of students')
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='studytime', y='G3')
plt.title('Final Grade by Weekly Study Time')
plt.xlabel('Study time category (1 = less than 2 hours; 4 = over 10 hours)')
plt.ylabel('Final grade (G3)')
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x='absences', y='G3', alpha=0.7)
plt.title('Absences vs Final Grade')
plt.xlabel('Number of absences')
plt.ylabel('Final grade (G3)')
plt.show()

## 5. Prepare features and target
`G3` is the value to predict. We intentionally remove `G1` and `G2`, because those grades may not be available when making an early prediction. This makes the task harder but more realistic.


In [ ]:
TARGET = 'G3'
excluded_columns = ['G3', 'G1', 'G2']
X = df.drop(columns=excluded_columns)
y = df[TARGET].astype(float)

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

print('Number of features:', X.shape[1])
print('Categorical features:', categorical_features)
print('Numerical features:', numerical_features)
print('\nTarget summary:')
display(y.describe())

## 6. Split data and preprocess
Numeric columns are imputed with the median and scaled. Categorical columns are imputed with the most frequent value and one-hot encoded. Preprocessing is fitted only on the training data to reduce data leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 7. Train and compare regression models


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    trained_models[name] = pipeline
    predictions[name] = pred
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2 Score': r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(results_df.round(3))
best_model_name = results_df.loc[0, 'Model']
best_model = trained_models[best_model_name]
print('Best model based on test RMSE:', best_model_name)

## 8. Visualize model performance


In [ ]:
best_predictions = predictions[best_model_name]

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=best_predictions, alpha=0.75)
plt.plot([0, 20], [0, 20], linestyle='--')
plt.xlim(0, 20)
plt.ylim(0, 20)
plt.xlabel('Actual final grade')
plt.ylabel('Predicted final grade')
plt.title(f'Actual vs Predicted Grades — {best_model_name}')
plt.show()

comparison = pd.DataFrame({
    'Actual grade': y_test.values,
    'Predicted grade': np.round(best_predictions, 2)
})
display(comparison.head(10))

## 9. Try a prediction for a student
The example below uses one row from the test set. It is a demonstration, not a real-world assessment of a student.


In [ ]:
sample_student = X_test.iloc[[0]]
predicted_grade = best_model.predict(sample_student)[0]
print('Sample student features:')
display(sample_student)
print(f'Predicted final grade: {predicted_grade:.2f} / 20')
print(f'Actual final grade: {y_test.iloc[0]:.2f} / 20')

## 10. Conclusion

- This project predicts final Mathematics grades from student and school-related information.
- It demonstrates data inspection, EDA, preprocessing, train/test splitting, regression, and evaluation using MAE, RMSE, and R².
- Lower MAE and RMSE indicate smaller prediction errors; R² closer to 1 generally indicates a better fit.
- Model scores can vary with the train/test split. The dataset is relatively small, and predictions should not be used to make consequential decisions about individual students.

**Dataset reference:** Cortez, P. (2008). *Student Performance*. UCI Machine Learning Repository. https://doi.org/10.24432/C5TG7T
